In [1]:
import pandas as pd
import requests
import io
from sqlalchemy import create_engine

# Database connection details
server = 'ANTHPC\\SQLEXPRESS'
database = 'NHL_01'
table_name = 'NHL_Gamelog'
url = 'https://moneypuck.com/moneypuck/playerData/careers/gameByGame/all_teams.csv'


In [2]:
# Step 1: Download CSV Data
try:
    response = requests.get(url)
    response.raise_for_status()
    file_object = io.StringIO(response.content.decode('utf-8'))
    data = pd.read_csv(file_object)
    print("CSV data loaded successfully.")
except requests.exceptions.RequestException as e:
    print(f"Error fetching CSV: {e}")
    exit()

CSV data loaded successfully.


In [3]:
# Step 2: Standardize gameId in the DataFrame
if 'gameId' in data.columns:
    data['gameId'] = data['gameId'].astype(str).str.strip().str.lower()
else:
    print("Column 'gameId' not found in the data.")
    exit()


In [4]:
# Step 3: Clean up team abbreviations
team_abbreviation_mapping = {
    'L.A': 'LAK',  # Example: Phoenix Coyotes became Arizona Coyotes
    'N.J': 'NJD',
    'S.J': 'SJS',
    'T.B': 'TBL'   # Example: Minnesota North Stars mapped to Minnesota Wild
    # Add more mappings as needed
}

# Define columns to clean
team_columns = ['team', 'playerTeam', 'opposingTeam']

# Apply the mapping
for col in team_columns:
    if col in data.columns:
        data[col] = data[col].replace(team_abbreviation_mapping)
    else:
        print(f"Column '{col}' not found in the data.")


In [5]:
# Step 4: Connect to SQL Server
connection_string = (
    f'mssql+pyodbc://{server}/{database}?driver=ODBC+Driver+17+for+SQL+Server&trusted_connection=yes'
)
try:
    engine = create_engine(connection_string)
    print("Connected to SQL Server successfully.")
except Exception as e:
    print(f"Error connecting to SQL Server: {e}")
    exit()


Connected to SQL Server successfully.


In [6]:
# Step 5: Deduplicate by gameId
try:
    # Fetch existing gameIds and standardize them
    existing_game_ids = pd.read_sql(f"SELECT DISTINCT gameId FROM {table_name}", con=engine)
    existing_game_ids['gameId'] = existing_game_ids['gameId'].astype(str).str.strip().str.lower()

    # Filter out rows with existing gameIds
    new_data = data[~data['gameId'].isin(existing_game_ids['gameId'])]
    print(f"New games identified: {len(new_data['gameId'].unique())}")

    # Append new data to the SQL table
    if not new_data.empty:
        new_data.to_sql(name=table_name, con=engine, if_exists='append', index=False)
        print(f"Data appended to table '{table_name}' successfully.")
    else:
        print("No new games to append.")

except Exception as e:
    print(f"Error during deduplication or data upload: {e}")


New games identified: 83
Data appended to table 'NHL_Gamelog' successfully.


In [7]:
# try:
#     data.to_sql(name=table_name, con=engine, if_exists= 'replace',index=False)
#     print(f"Data written to new table '{table_name}' successfully. ")

# except Exception as e:
#     print(f"error during table creation or data upload: {e}")
